# MolChat P2 — serve with vLLM (Colab GPU)

Serves the fine-tuned MolChat model with **vLLM**'s OpenAI-compatible server
and drives MolChat's RAG generation against it. Runtime: **GPU (T4)**.
vLLM needs CUDA, so this runs on Colab, not on a Mac.

In [ ]:
!pip -q install vllm openai
# Get the repo (private): set a token or upload the MolChat/ folder.
# import os; os.environ['GH_TOKEN']='...'; !git clone https://$GH_TOKEN@github.com/junghyun-han/MolChat.git
%cd MolChat

## 1. Start the vLLM OpenAI-compatible server
Serve the base instruct model, applying the LoRA adapter trained in
`p2_lora_qwen.ipynb` (or omit `--enable-lora` to serve the base model).

In [ ]:
import subprocess, time, os
cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
    '--port', '8000', '--max-model-len', '2048',
    # LoRA (optional): '--enable-lora', '--lora-modules', 'molchat=p2/out/molchat-qwen-lora',
]
server = subprocess.Popen(cmd)
time.sleep(60)  # wait for weights to load
print('vLLM server started on :8000')

In [ ]:
# 2. Sanity check: query vLLM directly with the OpenAI client
import openai
client = openai.OpenAI(base_url='http://localhost:8000/v1', api_key='x')
r = client.chat.completions.create(
    model='Qwen/Qwen2.5-0.5B-Instruct',
    messages=[{'role':'user','content':'What is the molecular weight of ethanol?'}],
    max_tokens=64)
print(r.choices[0].message.content)

In [ ]:
# 3. Drive MolChat RAG generation against the vLLM endpoint
import os
os.environ['MOLCHAT_GEN'] = 'local'
os.environ['MOLCHAT_GEN_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['MOLCHAT_GEN_MODEL'] = 'Qwen/Qwen2.5-0.5B-Instruct'
from molchat.rag_generate import answer_with_rag
res = answer_with_rag('Is aspirin likely to cross the blood-brain barrier?', molecule='aspirin')
print('backend:', res['backend'])
print('answer :', res['answer'])

Now a MolChat RAG answer is generated by a model **served with vLLM**. Record the
backend string and answer as evidence, then shut the server down with
`server.terminate()`.